<a href="https://colab.research.google.com/github/jessicalewinter/deep-learning-specialization/blob/main/course-notebooks/deep-architectures/Lab03_Limite_da_profundidade_aluno.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 3 · O limite da profundidade

**Arquiteturas Profundas e Representação de Dados** · Especialização em Deep Learning · CIn.AI / CIn-UFPE

Neste lab você mede o que acontece com o sinal e com o gradiente quando a rede fica funda, e reproduz, em pequeno, os dois resultados que levaram à ResNet.

- **Parte A (em aula, guiada, ~45 min):** inicialização. Uma torre de 10 camadas, sete jeitos de começar os pesos, suas previsões e os histogramas por camada. Fecha com 30 camadas: Xavier × He.
- **Parte B (em casa):** a degradação. CNNs "planas" com 8 e 20 camadas: qual erra menos no treino?
- **Parte C (em casa):** a solução existe. Construir uma rede funda que copia a rasa e treinar a funda começando pela identidade.
- **Desafio extra (opcional, não conta para a nota):** BatchNorm na rede plana, uma prévia da aula 5.

**Antes de começar**

1. *Arquivo → Salvar uma cópia no Drive*.
2. *Ambiente de execução → Alterar o tipo de ambiente de execução → GPU T4* (a parte A roda bem na CPU).
3. Rode as células na ordem. As células **Verificação** dizem `OK` ou `ERRO`.
4. No fim, a última célula gera o seu **código de conclusão** para o formulário de entrega.

## Parte A · em aula

### Preparação · configuração, dados e funções prontas

In [ ]:
# Célula 1 · Configuração
import os, time, json, hashlib, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import matplotlib.pyplot as plt

SEED = 2026
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
USE_AMP = DEVICE == 'cuda'
# Na CPU há menos imagens; mini-batches menores e mais épocas compensam o número menor de passos.
EPOCAS, BS = (6, 128) if DEVICE == 'cuda' else (8, 64)
print('GPU:', torch.cuda.get_device_name(0)) if DEVICE == 'cuda' else print('Sem GPU: as partes B e C usam 10 mil imagens na CPU.')

CHECKS = {}
def check(chave, condicao, ok, erro):
    """Registra uma verificação e mostra OK ou ERRO."""
    CHECKS[chave] = bool(condicao)
    print(('OK    ' + ok) if condicao else ('ERRO  ' + erro))

In [ ]:
# Célula 2 · CIFAR-10 (como nos labs 1 e 2)
ESPELHO_URL = None   # "https://drive.google.com/file/d/1fBXTy06cRH4M4fugrFSQOwqvhjOfAqDS/view?usp=sharing"

t0 = time.time()
if ESPELHO_URL:
    if 'drive.google.com' in ESPELHO_URL:
        import gdown                      # já vem no Colab; baixa arquivos grandes do Google Drive
        gdown.download(ESPELHO_URL, 'cifar10.npz', quiet=False, fuzzy=True)
    else:
        import urllib.request
        urllib.request.urlretrieve(ESPELHO_URL, 'cifar10.npz')
    d = np.load('cifar10.npz')
    tr_x, tr_y, te_x, te_y = d['x_train'], d['y_train'], d['x_test'], d['y_test']
else:
    tr = torchvision.datasets.CIFAR10(root='dados', train=True, download=True)
    te = torchvision.datasets.CIFAR10(root='dados', train=False, download=True)
    tr_x, tr_y, te_x, te_y = tr.data, np.array(tr.targets), te.data, np.array(te.targets)

N_TREINO, N_TESTE = (len(tr_x), len(te_x)) if DEVICE == 'cuda' else (10_000, 2_000)
def para_tensor(x, y, n):
    return torch.from_numpy(x[:n]).permute(0, 3, 1, 2).float().div(255), torch.from_numpy(np.asarray(y[:n])).long()
x_treino, y_treino = para_tensor(tr_x, tr_y, N_TREINO)
x_teste, y_teste = para_tensor(te_x, te_y, N_TESTE)
MEDIA = x_treino.mean(dim=(0, 2, 3), keepdim=True); DESVIO = x_treino.std(dim=(0, 2, 3), keepdim=True)
x_treino = ((x_treino - MEDIA) / DESVIO).to(DEVICE); x_teste = ((x_teste - MEDIA) / DESVIO).to(DEVICE)
y_treino, y_teste = y_treino.to(DEVICE), y_teste.to(DEVICE)
print(f'Treino: {tuple(x_treino.shape)} · Teste: {tuple(x_teste.shape)} · {time.time() - t0:.0f}s')
check('dados', x_treino.shape[1:] == (3, 32, 32) and len(x_treino) == N_TREINO, 'dados carregados', 'formato inesperado: rode a célula 2 de novo')

In [ ]:
# Célula 3 · Funções prontas: avaliar e treinar
def avaliar(model, x, y, bs=1000):
    model.eval(); acertos = 0
    with torch.no_grad(), torch.autocast(device_type=DEVICE, dtype=torch.float16, enabled=USE_AMP):
        for i in range(0, len(x), bs):
            acertos += (model(x[i:i + bs]).argmax(1) == y[i:i + bs]).sum().item()
    return acertos / len(x)

def treinar(model, epocas=EPOCAS, bs=BS, lr=1e-3, otimizador='adam', amp=True, nome='modelo', log_cada=20):
    """Treina no CIFAR-10. Devolve um dicionário com a curva da perda, o acerto no treino e no teste e o tempo."""
    torch.manual_seed(SEED); model = model.to(DEVICE); usar_amp = USE_AMP and amp
    opt = (torch.optim.Adam(model.parameters(), lr=lr) if otimizador == 'adam'
           else torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9))
    scaler = torch.amp.GradScaler(enabled=usar_amp); t0 = time.time(); curva, acum, passo = [], [], 0
    for ep in range(epocas):
        model.train(); ordem = torch.randperm(len(x_treino), device=DEVICE)
        for i in range(0, len(x_treino), bs):
            b = ordem[i:i + bs]
            with torch.autocast(device_type=DEVICE, dtype=torch.float16, enabled=usar_amp):
                perda = F.cross_entropy(model(x_treino[b]).float(), y_treino[b])
            opt.zero_grad(set_to_none=True); scaler.scale(perda).backward(); scaler.step(opt); scaler.update()
            acum.append(perda.item()); passo += 1
            if passo % log_cada == 0: curva.append((passo, float(np.mean(acum)))); acum = []
    r = {'modelo': model, 'curva': curva, 'acc_treino': avaliar(model, x_treino[:10_000], y_treino[:10_000]),
         'acc_teste': avaliar(model, x_teste, y_teste), 'tempo': time.time() - t0}
    print(f"{nome}: acerto no treino {r['acc_treino']:.1%} · no teste {r['acc_teste']:.1%} · {r['tempo']:.0f}s")
    return r

### Passo 1 · Uma torre de 10 camadas e quatro jeitos de sortear os pesos (10 min)

A torre é uma rede densa: a imagem achatada (3.072 números) passa por 10 camadas de 512 neurônios. Cada neurônio soma `n` entradas multiplicadas por pesos, então, se entradas e pesos são independentes e têm média zero:

**Var(saída) = n · Var(w) · Var(entrada)**

Para o sinal manter o tamanho, `n · Var(w)` precisa ser 1. Isso dá a inicialização de **Xavier** (Glorot & Bengio, 2010): desvio dos pesos = √(1/n). A ReLU zera metade das entradas e corta a variância pela metade; para compensar, **He** (He et al., 2015) usa desvio √(2/n).

Complete `inicializar` com o desvio de cada método. `n` é o número de entradas da camada (`in_features`). O método `'padrao'` não mexe em nada: fica o sorteio que o próprio PyTorch faz.

In [ ]:
# Célula 4 · A torre e a inicialização
ATIVACOES = {'tanh': nn.Tanh, 'sigmoide': nn.Sigmoid, 'relu': nn.ReLU}

def torre(camadas=10, largura=512, ativacao='tanh', entradas=3 * 32 * 32):
    """Rede densa: Flatten → (Linear + ativação) × camadas → Linear de saída com 10 classes."""
    modulos, n = [nn.Flatten()], entradas
    for _ in range(camadas):
        modulos += [nn.Linear(n, largura), ATIVACOES[ativacao]()]
        n = largura
    modulos.append(nn.Linear(n, 10))
    return nn.Sequential(*modulos)

def inicializar(modelo, metodo):
    """metodo: 'pequena' (desvio 0,01), 'grande' (desvio 1), 'xavier', 'he' ou 'padrao' (não mexe)."""
    if metodo == 'padrao':
        return modelo
    for camada in modelo.modules():
        if isinstance(camada, nn.Linear):
            n = camada.in_features
            # TODO: escolha o desvio padrão dos pesos para cada método
            raise NotImplementedError('Complete o TODO acima')
            nn.init.normal_(camada.weight, 0.0, desvio)
            nn.init.zeros_(camada.bias)
    return modelo

**Verificação.**

In [ ]:
_t = torre(2, 512)
_ok = True
for _m, _alvo in [('pequena', 0.01), ('grande', 1.0), ('xavier', (1 / 512) ** .5), ('he', (2 / 512) ** .5)]:
    torch.manual_seed(0); inicializar(_t, _m); _s = _t[3].weight.std().item()
    print(f'{_m:8s} desvio dos pesos da camada 2: {_s:.4f} (esperado {_alvo:.4f})'); _ok &= abs(_s / _alvo - 1) < 0.02
check('init', _ok, 'os quatro desvios estão certos', 'algum desvio não bate: confira a fórmula de cada método')

### Passo 2 · Preveja antes de ver (5 min)

Vamos passar 1.000 imagens por sete torres de 10 camadas. Para cada uma, preveja o que acontece com as ativações da **camada 10**, comparadas às da camada 1:

- `'some'`: o desvio cai para menos de 1% do da camada 1;
- `'encolhe'`: cai bastante, ficando entre 1% e 30%;
- `'mantem'`: fica com pelo menos 30%;
- `'satura'`: a maioria das ativações tanh encosta em −1 ou +1.

A nota não depende de acertar; vale pela previsão registrada **antes** de rodar a célula 6.

In [ ]:
# Célula 5 · Suas previsões
CONFIGS = [('tanh', 'pequena'), ('tanh', 'grande'), ('tanh', 'xavier'), ('sigmoide', 'xavier'),
           ('relu', 'xavier'), ('relu', 'he'), ('relu', 'padrao')]
PREVISAO = {c: '' for c in CONFIGS}   # preencha: PREVISAO[('tanh', 'pequena')] = 'some', etc.
check('previsao', all(PREVISAO.get(c) in ('some', 'encolhe', 'mantem', 'satura') for c in CONFIGS),
      'sete previsões registradas', "preencha as sete previsões com 'some', 'encolhe', 'mantem' ou 'satura'")

### Passo 3 · O sinal e o gradiente, camada por camada (15 min)

A célula abaixo, pronta, usa *ganchos* (`register_forward_hook`) para guardar a saída de cada ativação e, depois do `backward`, lê o gradiente dos pesos de cada camada. Rode e compare com as suas previsões.

In [ ]:
# Célula 6 · Perfil das sete torres
X_PERFIL, Y_PERFIL = x_treino[:1000], y_treino[:1000]

def perfil(modelo):
    """Ativações e desvio de cada camada (para a frente) e norma do gradiente dos pesos (para trás)."""
    modelo = modelo.to(DEVICE); ativacoes = []
    ganchos = [m.register_forward_hook(lambda mod, ent, sai: ativacoes.append(sai.detach()))
               for m in modelo if isinstance(m, (nn.Tanh, nn.Sigmoid, nn.ReLU))]
    modelo.zero_grad()
    F.cross_entropy(modelo(X_PERFIL), Y_PERFIL).backward()
    for g in ganchos: g.remove()
    lineares = [m for m in modelo if isinstance(m, nn.Linear)][:-1]
    return {'ativacoes': [a.float().cpu() for a in ativacoes], 'desvio': [a.float().std().item() for a in ativacoes],
            'gradiente': [l.weight.grad.norm().item() for l in lineares]}

def o_que_aconteceu(p, ativacao):
    d1, d10 = p['desvio'][0], p['desvio'][-1]
    if ativacao == 'tanh' and (p['ativacoes'][-1].abs() > 0.95).float().mean() > 0.5: return 'satura'
    if d10 < 0.01 * d1: return 'some'
    if d10 < 0.3 * d1: return 'encolhe'
    return 'mantem'

PERFIS, ACERTOS = {}, 0
print(f"{'torre':20s} {'desvio c1':>10s} {'desvio c10':>11s} {'grad c1 ÷ c10':>14s}  aconteceu  previsão")
for ativ, init in CONFIGS:
    torch.manual_seed(SEED); p = perfil(inicializar(torre(10, 512, ativ), init)); PERFIS[(ativ, init)] = p
    real = o_que_aconteceu(p, ativ); ACERTOS += real == PREVISAO[(ativ, init)]
    print(f"{ativ + ' + ' + init:20s} {p['desvio'][0]:10.3g} {p['desvio'][-1]:11.3g} {p['gradiente'][0] / p['gradiente'][-1]:14.3g}  {real:9s}  {PREVISAO[(ativ, init)]}")
print(f'\nVocê acertou {ACERTOS} de 7. Cole esse número no formulário da turma.')

# a grade de histogramas: uma linha por torre, uma coluna por camada
fig, eixos = plt.subplots(len(CONFIGS), 10, figsize=(16, 1.35 * len(CONFIGS)), sharey='row')
for i, (ativ, init) in enumerate(CONFIGS):
    p = PERFIS[(ativ, init)]
    for l in range(10):
        a = p['ativacoes'][l].flatten().numpy(); ax = eixos[i, l]
        faixa = (-1, 1) if ativ == 'tanh' else (0, 1) if ativ == 'sigmoide' else (0, max(a.max(), 1e-6))
        ax.hist(a, bins=30, range=faixa, color='#3C78D8'); ax.set_xticks([]); ax.set_yticks([])
        ax.set_title(f'{p["desvio"][l]:.2g}', fontsize=8)
        if l == 0: ax.set_ylabel(f'{ativ}\n{init}', fontsize=8, rotation=0, ha='right', va='center')
fig.suptitle('Ativações por camada (o número é o desvio padrão)', y=1.0); plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(9, 3.6))
for (ativ, init), p in PERFIS.items():
    ax.semilogy(range(1, 11), p['gradiente'], 'o-', label=f'{ativ} + {init}')
ax.set_xlabel('camada'); ax.set_ylabel('norma do gradiente dos pesos'); ax.legend(fontsize=8, ncol=2); ax.grid(alpha=.3); plt.show()
check('perfil', len(PERFIS) == 7 and o_que_aconteceu(PERFIS[('relu', 'he')], 'relu') == 'mantem',
      'perfis das sete torres medidos', "rode de novo: a torre ReLU + He deveria manter o sinal; confira o 'he' na célula 4")

**Responda (2 a 4 linhas):** o padrão do PyTorch sorteia os pesos de uma distribuição uniforme entre −1/√n e +1/√n, que tem variância 1/(3n). Com ReLU, por quanto o desvio das ativações é multiplicado a cada camada? Compare com o que a tabela mostrou. (Dica: com ReLU, Var(saída) = n · Var(w) · Var(entrada) / 2.)

In [ ]:
# Célula 7 · Sua resposta
RESPOSTA_A = """
escreva aqui
"""
_t = RESPOSTA_A.replace('escreva aqui', '').strip()
check('resposta_a', len(_t) >= 80, 'resposta registrada', 'escreva pelo menos 2 linhas')

### Passo 4 · 30 camadas: Xavier trava, He treina (10 min)

He et al. (2015) mostraram que uma rede de 30 camadas com ReLU não sai do lugar com a inicialização de Xavier e treina com a deles. Vamos reproduzir em pequeno: torres de 30 camadas × 256 neurônios, SGD, 2 épocas. A linha tracejada é a perda de quem chuta: ln 10 ≈ 2,30.

In [ ]:
# Célula 8 · Xavier × He em 30 camadas
TRINTA = {}
for init in ['xavier', 'he']:
    torch.manual_seed(SEED)
    TRINTA[init] = treinar(inicializar(torre(30, 256, 'relu'), init), epocas=2, lr=0.01, otimizador='sgd', amp=False, nome=f'30 camadas, {init}')
fig, ax = plt.subplots(figsize=(8, 3.4))
for init, r in TRINTA.items():
    ax.plot(*zip(*r['curva']), label=init)
ax.axhline(np.log(10), ls='--', c='gray'); ax.set_xlabel('passo'); ax.set_ylabel('perda no treino'); ax.legend(); plt.show()
_fim = {k: np.mean([v for _, v in r['curva'][-5:]]) for k, r in TRINTA.items()}
print(f"perda no fim: Xavier {_fim['xavier']:.3f} · He {_fim['he']:.3f}")
check('trinta', _fim['he'] < _fim['xavier'] - 0.1, 'com He a rede de 30 camadas treina; com Xavier, mal sai do chute',
      'o resultado não bateu: confira o desvio do He na célula 4')

## Parte B · em casa: mais fundo, pior

A CNN "plana" abaixo tem o formato das redes do artigo da ResNet para o CIFAR-10: uma convolução de entrada, três estágios de `2n` convoluções 3×3 (16, 32 e 64 filtros; o primeiro de cada estágio novo usa stride 2), média global e a camada de saída. São `6n + 2` camadas com pesos. Não há atalhos nem BatchNorm, e todas as convoluções começam com He.

Treine as versões com 8 (`n = 1`) e 20 camadas (`n = 3`). Na GPU, a célula também treina a de 56 (`n = 9`). Olhe o acerto **no treino**.

In [ ]:
# Célula 9 · A CNN plana (pronta)
def cnn_plana(n, bn=False, init='he'):
    """6n + 2 camadas. init: 'he' ou 'identidade' (as convoluções que não mudam o formato começam copiando a entrada)."""
    def bloco(cin, cout, stride):
        conv = nn.Conv2d(cin, cout, 3, stride=stride, padding=1, bias=not bn)
        if init == 'identidade' and cin == cout and stride == 1:
            nn.init.dirac_(conv.weight)
        else:
            nn.init.kaiming_normal_(conv.weight, nonlinearity='relu')
        if conv.bias is not None: nn.init.zeros_(conv.bias)
        return [conv] + ([nn.BatchNorm2d(cout)] if bn else []) + [nn.ReLU()]
    camadas, cin = bloco(3, 16, 1), 16
    for estagio, c in enumerate([16, 32, 64]):
        for i in range(2 * n):
            camadas += bloco(cin, c, 2 if (estagio > 0 and i == 0) else 1); cin = c
    camadas += [nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(cin, 10)]
    return nn.Sequential(*camadas)

def contar_camadas(m):
    return sum(isinstance(x, (nn.Conv2d, nn.Linear)) for x in m.modules())
for n in [1, 3, 9]:
    m = cnn_plana(n); _p = f'{sum(p.numel() for p in m.parameters()):,}'.replace(',', '.')
    print(f'n = {n}: {contar_camadas(m)} camadas, {_p} parâmetros')

In [ ]:
# Célula 10 · Treinar 8 e 20 camadas (e 56 na GPU)
RODAR_56 = DEVICE == 'cuda'
RESULTADOS_B = {}
for n in ([1, 3, 9] if RODAR_56 else [1, 3]):
    torch.manual_seed(SEED); RESULTADOS_B[6 * n + 2] = treinar(cnn_plana(n), nome=f'{6 * n + 2} camadas')
fig, ax = plt.subplots(figsize=(8, 3.4))
for camadas, r in RESULTADOS_B.items():
    ax.plot(*zip(*r['curva']), label=f'{camadas} camadas')
ax.axhline(np.log(10), ls='--', c='gray'); ax.set_xlabel('passo'); ax.set_ylabel('perda no treino'); ax.legend(); plt.show()
print('Degradação apareceu?', 'sim' if RESULTADOS_B[20]['acc_treino'] < RESULTADOS_B[8]['acc_treino'] else 'não nesta rodada')
check('tabela_b', 8 in RESULTADOS_B and 20 in RESULTADOS_B, 'redes de 8 e 20 camadas treinadas', 'rode a célula de novo')

**Responda (3 a 5 linhas):** a rede de 20 camadas tem mais parâmetros e pode representar tudo o que a de 8 representa. Por que ela erra mais *no treino*? Isso é overfitting? Justifique com os números.

In [ ]:
# Célula 11 · Sua resposta
RESPOSTA_B = """
escreva aqui
"""
_t = RESPOSTA_B.replace('escreva aqui', '').strip()
check('resposta_b', len(_t) >= 120, 'resposta registrada', 'escreva pelo menos 3 linhas')

## Parte C · em casa: a solução existe

He et al. (2016) observaram que a rede funda **pode** ser pelo menos tão boa quanto a rasa: basta copiar as camadas da rasa e completar com camadas que não fazem nada, a identidade. Vamos construir essa rede.

Uma convolução 3×3 copia a entrada quando o peso central do filtro do canal `c` para o canal `c` vale 1 e todos os outros valem 0: é o que `nn.init.dirac_` faz. Como a entrada dessas camadas já passou por uma ReLU (é ≥ 0), a ReLU depois delas também não muda nada.

Complete `aprofundar`: depois de cada ReLU da rede rasa (menos a primeira), insira `extras` pares de convolução identidade + ReLU.

In [ ]:
# Célula 12 · Construir a rede funda a partir da rasa
def aprofundar(rasa, extras):
    camadas, relus = [], 0
    for modulo in rasa:
        camadas.append(modulo)
        if isinstance(modulo, nn.ReLU):
            relus += 1
            if relus == 1:
                continue                      # não mexe na camada de entrada
            canais = camadas[-2].out_channels
            for _ in range(extras):
                conv = nn.Conv2d(canais, canais, 3, padding=1)
                # TODO: faça a convolução copiar a entrada (pesos identidade e viés zero)
                raise NotImplementedError('Complete o TODO acima')
                camadas += [conv, nn.ReLU()]
    return nn.Sequential(*camadas).to(DEVICE)

rasa = RESULTADOS_B[8]['modelo']
funda = aprofundar(rasa, 8)
rasa.eval(); funda.eval()
with torch.no_grad():
    DIFERENCA = (rasa(x_teste[:1000]) - funda(x_teste[:1000])).abs().max().item()
ACC_FUNDA = avaliar(funda, x_treino[:10_000], y_treino[:10_000])
print(f'rede construída: {contar_camadas(funda)} camadas · diferença máxima nas saídas: {DIFERENCA:.2g}')
print(f"acerto no treino: rasa {RESULTADOS_B[8]['acc_treino']:.1%} · construída {ACC_FUNDA:.1%}")

**Verificação.**

In [ ]:
check('aprofundar', contar_camadas(funda) == 56 and DIFERENCA < 1e-3,
      'a rede de 56 camadas construída dá as mesmas saídas que a de 8', 'as saídas mudaram: confira os pesos e o viés da convolução identidade')

A solução existe, mas o treino da rede de 20 (ou 56) camadas não a encontrou. E se, em vez de sortear as camadas extras com He, elas **começarem** como identidade? Treine a CNN plana de 20 camadas com `init='identidade'` e compare com a da parte B.

In [ ]:
# Célula 13 · Começar pela identidade
RESULTADOS_C = {'20 camadas, He': RESULTADOS_B[20]}
torch.manual_seed(SEED)
RESULTADOS_C['20 camadas, identidade'] = treinar(cnn_plana(3, init='identidade'), nome='20 camadas, identidade')
print()
for nome, r in [('8 camadas, He', RESULTADOS_B[8])] + list(RESULTADOS_C.items()):
    print(f"{nome:24s} treino {r['acc_treino']:.1%} · teste {r['acc_teste']:.1%}")
check('tabela_c', len(RESULTADOS_C) == 2, 'comparação registrada', 'rode a célula de novo')

**Responda (3 a 5 linhas):** por que começar as camadas extras como identidade ajuda tanto? Pense no que o gradiente encontra no caminho de volta logo no primeiro passo. O que isso sugere para o desenho de uma arquitetura profunda?

In [ ]:
# Célula 14 · Sua resposta
RESPOSTA_C = """
escreva aqui
"""
_t = RESPOSTA_C.replace('escreva aqui', '').strip()
check('resposta_c', len(_t) >= 120, 'resposta registrada', 'escreva pelo menos 3 linhas')

## Desafio extra (opcional) · BatchNorm na rede plana

Na aula 5 veremos a BatchNorm. O artigo da ResNet treinou as redes planas **com** BatchNorm e ainda assim a de 56 camadas ficou pior que a de 20. Coloque `bn=True` e veja se a degradação some na sua rodada.

In [ ]:
# Célula 15 · BatchNorm (opcional)
FAZER_DESAFIO = False
if FAZER_DESAFIO:
    for n in ([1, 9] if DEVICE == 'cuda' else [1, 3]):
        torch.manual_seed(SEED); treinar(cnn_plana(n, bn=True), nome=f'{6 * n + 2} camadas com BatchNorm')